# Healthcare Predictive Modeling: Consolidated Pipeline

This notebook consolidates the machine learning pipeline for predicting chronic conditions from patient features, focusing on XGBoost models with both full and ablated feature sets.

In [ ]:
# Setup: Imports and Configuration

import pandas as pd
import numpy as np
import snowflake.connector
import getpass
import joblib
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    classification_report
)
from xgboost import XGBClassifier

# Snowflake Connection Settings (using getpass for secure password entry)
SNOWFLAKE_ACCOUNT = 'BSTPMLK-RA21160'
SNOWFLAKE_USER = 'AHMEDSAMI'
SNOWFLAKE_PASSWORD = getpass.getpass('Enter Snowflake Password: ')
SNOWFLAKE_ROLE = 'ACCOUNTADMIN'
SNOWFLAKE_WAREHOUSE = 'COMPUTE_WH'
SNOWFLAKE_DATABASE = 'HEALTHCARE_DB'
SNOWFLAKE_SCHEMA = 'ML'

# Global Configuration Constants
TARGETS = ['Diabetes', 'Hypertension', 'Coronary Heart Disease', 'Stroke', 'Asthma']
CATEGORICAL_COLS = ['GENDER', 'RACE', 'ETHNICITY', 'MARITAL']

# Full set of numeric features
NUMERIC_COLS_FULL = [
    'AGE_AT_INDEX', 'AVG_HEIGHT', 'AVG_BMI', 'AVG_WEIGHT', 'AVG_DIASTOLIC_BP',
    'AVG_GLUCOSE', 'AVG_SYSTOLIC_BP', 'AVG_CHOLESTEROL',
    'OBSERVATION_COUNT', 'DISTINCT_CONDITION_COUNT',
    'DISTINCT_MEDICATION_COUNT', 'TOTAL_ENCOUNTERS',
    'HAS_GLUCOSE_READING', 'HAS_CHOLESTEROL_READING'
]

# Ablated set of numeric features (removed potentially leaky 'count' features)
NUMERIC_COLS_ABLATED = [
    'AGE_AT_INDEX', 'AVG_HEIGHT', 'AVG_BMI', 'AVG_WEIGHT', 'AVG_DIASTOLIC_BP',
    'AVG_GLUCOSE', 'AVG_SYSTOLIC_BP', 'AVG_CHOLESTEROL',
    'HAS_GLUCOSE_READING', 'HAS_CHOLESTEROL_READING'
]

# XGBoost Model Hyperparameters
XGB_MODEL_PARAMS = {
    'random_state': 42,
    'n_jobs': -1,
    'eval_metric': 'aucpr',
    'n_estimators': 300,
    'max_depth': 5,
    'learning_rate': 0.05
}

Enter Snowflake Password: ··········


In [ ]:
# Data Loading Function

def load_data(target_name: str) -> pd.DataFrame:
    """Loads patient feature data for a specific target from Snowflake."""
    print(f"Loading data for target: {target_name}...")

    # Establish a fresh connection for each data load
    ctx = snowflake.connector.connect(
        account=SNOWFLAKE_ACCOUNT, user=SNOWFLAKE_USER, password=SNOWFLAKE_PASSWORD,
        role=SNOWFLAKE_ROLE, warehouse=SNOWFLAKE_WAREHOUSE, database=SNOWFLAKE_DATABASE, schema=SNOWFLAKE_SCHEMA
    )

    try:
        # Set session timeout for potentially long queries
        cur = ctx.cursor()
        cur.execute("ALTER SESSION SET STATEMENT_TIMEOUT_IN_SECONDS = 300;")

        query = f"SELECT * FROM {SNOWFLAKE_SCHEMA}.PATIENT_FEATURES_TARGETS_LONG WHERE TARGET_NAME = '{target_name}'"
        cur.execute(query)
        df_target = cur.fetch_pandas_all()
        cur.close()
    finally:
        ctx.close()

    print(f"Loaded {len(df_target)} rows for {target_name}.")

    # Apply age safety check
    # This assumes 'AGE_AT_INDEX' is always present in the loaded DataFrame
    df_target['AGE_AT_INDEX'] = df_target['AGE_AT_INDEX'].apply(lambda x: x / 365.25 if pd.notnull(x) and x > 200 else x)

    return df_target

In [ ]:
# Reusable Training and Evaluation Function

def train_evaluate_model(
    df_target: pd.DataFrame,
    numeric_features_to_use: list,
    categorical_features_to_use: list,
    model_instance,
    model_name: str,
    target_name: str,
    feature_set_name: str
) -> tuple:
    """
    Trains and evaluates a given model with specific feature sets.
    Returns a tuple: (results_dict, fitted_pipeline, feature_importance_df)
    """

    print(f"\n--- Training {model_name} with {feature_set_name} features for {target_name} ---")

    # Define X and y for the current target
    # Note: 'LABEL' is the target column, others are features or metadata
    X = df_target.drop(columns=['PATIENT_ID', 'TARGET_NAME', 'LABEL', 'INDEX_DATE'])
    y = df_target['LABEL']

    # Select only the numeric features specified for the current feature set
    X = X[numeric_features_to_use + categorical_features_to_use]

    if y.nunique() < 2:
        print(f"Skipping {target_name} - only one class present in data after feature selection.")
        return None, None, None

    # 80/20 Train-Test Split (stratified to maintain class balance)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Calculate dynamic scale_pos_weight for XGBoost (Handling Class Imbalance)
    # This needs to be applied to the model_instance before fitting
    spw = (y_train == 0).sum() / (y_train == 1).sum()
    if hasattr(model_instance, 'set_params'):
        model_instance.set_params(scale_pos_weight=spw)

    # Define the preprocessor for the current feature set
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='median'), numeric_features_to_use),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_to_use)
        ],
        remainder='drop' # Drop columns not specified in transformers
    )

    # Build the pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model_instance)
    ])

    # Fit the pipeline
    pipeline.fit(X_train, y_train)

    # Predictions & Core Metrics
    y_proba_train = pipeline.predict_proba(X_train)[:, 1]
    y_proba_test = pipeline.predict_proba(X_test)[:, 1]

    # Threshold Tuning for F1-Score
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_test)
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    best_idx = f1_scores.argmax()

    optimal_threshold = thresholds[best_idx]
    y_pred_optimal_train = (y_proba_train >= optimal_threshold).astype(int)
    y_pred_optimal_test = (y_proba_test >= optimal_threshold).astype(int)

    # --- Overfitting Check: Train vs Test Accuracy/AUC (using optimal threshold for accuracy) ---
    train_acc = accuracy_score(y_train, y_pred_optimal_train)
    test_acc = accuracy_score(y_test, y_pred_optimal_test)
    train_auc = roc_auc_score(y_train, y_proba_train)
    test_auc = roc_auc_score(y_test, y_proba_test)

    print(f"Train Accuracy (Optimal Thresh): {train_acc:.4f}   Test Accuracy (Optimal Thresh): {test_acc:.4f}")
    print(f"Train ROC-AUC:  {train_auc:.4f}   Test ROC-AUC:  {test_auc:.4f}")

    roc_auc = test_auc
    pr_auc = average_precision_score(y_test, y_proba_test)
    prevalence = y_test.mean()
    lift = pr_auc / prevalence if prevalence > 0 else 0

    print(f"ROC-AUC: {roc_auc:.4f}  |  PR-AUC: {pr_auc:.4f}  |  Lift: {lift:.1f}x")
    print(f"Optimal F1-Score: {f1_scores[best_idx]:.3f} (Threshold: {optimal_threshold:.3f})")
    print("Classification Report (at optimal F1 threshold):")
    print(classification_report(y_test, y_pred_optimal_test))

    # Feature Importance Extraction
    feature_names_out = pipeline.named_steps['preprocessor'].get_feature_names_out()
    importances = pipeline.named_steps['classifier'].feature_importances_
    feature_importance_df = pd.DataFrame({'Feature': feature_names_out, 'Importance': importances})
    feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

    results_dict = {
        'Target': target_name,
        'Feature_Set': feature_set_name,
        'Model': model_name,
        'ROC-AUC': round(roc_auc, 4),
        'PR-AUC': round(pr_auc, 4),
        'Lift': round(lift, 1),
        'Optimal_F1_Score': round(f1_scores[best_idx], 3),
        'Optimal_Threshold': round(optimal_threshold, 3),
        'Train_Accuracy': round(train_acc, 4),
        'Test_Accuracy': round(test_acc, 4),
        'Train_ROC-AUC': round(train_auc, 4)
    }

    return results_dict, pipeline, feature_importance_df

## Why Mortality Was Dropped as a Target

Initially, 'Mortality' was considered as a potential target variable. However, upon deeper analysis, it was decided to drop it from the scope of this project for several reasons:

1.  **Complexity of Causality:** Mortality is often the ultimate outcome of multiple interacting health conditions, lifestyle factors, and acute events. Attributing it directly to specific chronic conditions without extensive causal modeling is challenging and can lead to misleading conclusions.
2.  **Data Scarcity for Predictive Power:** While death is a clear outcome, the specific events leading to it are diverse. Building a predictive model for overall mortality based solely on the chronic condition features available was deemed less impactful than focusing on specific disease predictions.
3.  **Ethical and Practical Considerations:** Predicting mortality can have profound ethical implications and requires very careful interpretation. For this project's scope, focusing on the prevention and early detection of treatable chronic conditions offers more immediate and actionable clinical utility.

## Understanding Target Leakage

Target leakage is a critical issue in machine learning where information about the target variable, which would not be available in a real-world prediction scenario, is inadvertently used during model training. This leads to artificially inflated performance metrics that do not reflect the model's true predictive power.

In this project, two main types of target leakage were identified:

1.  **Temporal Leakage:** This occurs when features are created using data collected *after* the `INDEX_DATE` (the date of diagnosis for the target condition or a surrogate event). For instance, if a feature reflects treatments or symptoms that only appear after a diagnosis, using it to predict that diagnosis is temporal leakage. The `INDEX_DATE` was introduced to establish a clear timeline, ensuring that all features are derived from data *before* the patient's index date for the condition.

2.  **Definitional Leakage:** This happens when features are inherently tied to the definition of the target variable. For example, features like `OBSERVATION_COUNT`, `DISTINCT_CONDITION_COUNT`, `DISTINCT_MEDICATION_COUNT`, and `TOTAL_ENCOUNTERS` often skyrocket *after* a patient receives a diagnosis because they begin receiving more medical attention, diagnoses, and prescriptions. If these counts are used to predict the diagnosis itself, they are essentially 'leaking' the diagnosis because they are a consequence of it. This was the primary reason for creating an 'ablated' feature set that excludes these potentially leaky count-based features.

## Index Dates, Incidence-Density Sampling, and Buffer Windows

To combat the identified target leakage and build clinically relevant models, several key methodological steps were introduced:

1.  **Index Dates:** A specific `INDEX_DATE` was established for each patient and target condition. This date represents the first documented instance of a diagnosis or a significant event related to the condition. All features used for prediction are strictly derived from patient data *prior to* this `INDEX_DATE`. This ensures that the model only learns from information that would be available at the time of prediction.

2.  **Incidence-Density Sampling:** This technique was employed to select control patients. For each case (a patient with a diagnosis), a control patient is selected who is matched on certain criteria (e.g., age, gender) and who has *not* developed the condition by the case's `INDEX_DATE`. This helps to create a more realistic control group and reduces bias.

3.  **Buffer Windows:** In some contexts, a 'buffer window' is used. This refers to a period immediately preceding the `INDEX_DATE` during which certain events might be excluded from feature calculation to avoid very subtle forms of leakage (e.g., prodromal symptoms that are effectively part of the diagnostic process but appear just before the formal diagnosis). While not explicitly implemented with a strict exclusion window in the current feature set, the principle underpins the careful selection of pre-index features.

In [ ]:
# Experiment Runner

all_results = [] # To store evaluation metrics for all experiments
all_feature_importances = {} # To store feature importances for all experiments
full_feature_pipelines = {} # To store fitted pipelines for full feature set for saving

for target_name in TARGETS:
    # Load data for the current target
    df_target = load_data(target_name)

    if df_target is None or df_target.empty:
        print(f"No data loaded for {target_name}, skipping.")
        continue

    # --- Experiment with FULL Feature Set ---
    print(f"\n--- Running FULL feature set experiment for {target_name} ---")
    xgb_full = XGBClassifier(**XGB_MODEL_PARAMS) # Initialize model for each experiment

    results_full, pipeline_full, fi_df_full = train_evaluate_model(
        df_target,
        NUMERIC_COLS_FULL,
        CATEGORICAL_COLS,
        xgb_full,
        'XGBoost',
        target_name,
        'Full Features'
    )

    if results_full:
        all_results.append(results_full)
        all_feature_importances[f'{target_name}_Full'] = fi_df_full
        full_feature_pipelines[target_name] = pipeline_full # Store for saving

    # --- Experiment with ABLATED Feature Set ---
    print(f"\n--- Running ABLATED feature set experiment for {target_name} ---")
    xgb_ablated = XGBClassifier(**XGB_MODEL_PARAMS) # Initialize model for each experiment

    results_ablated, pipeline_ablated, fi_df_ablated = train_evaluate_model(
        df_target,
        NUMERIC_COLS_ABLATED,
        CATEGORICAL_COLS,
        xgb_ablated,
        'XGBoost',
        target_name,
        'Ablated Features'
    )

    if results_ablated:
        all_results.append(results_ablated)
        all_feature_importances[f'{target_name}_Ablated'] = fi_df_ablated

# Convert all results to a DataFrame for comparison
final_comparison_df = pd.DataFrame(all_results)

print("\n\n=== FINAL MODEL COMPARISON ===")
print(final_comparison_df.to_string())

Loading data for target: Diabetes...
Loaded 118116 rows for Diabetes.

--- Running FULL feature set experiment for Diabetes ---

--- Training XGBoost with Full Features features for Diabetes ---
Train Accuracy (Optimal Thresh): 0.9232   Test Accuracy (Optimal Thresh): 0.9237
Train ROC-AUC:  0.9580   Test ROC-AUC:  0.9466
ROC-AUC: 0.9466  |  PR-AUC: 0.5070  |  Lift: 8.3x
Optimal F1-Score: 0.560 (Threshold: 0.836)
Classification Report (at optimal F1 threshold):
              precision    recall  f1-score   support

           0       0.99      0.93      0.96     22180
           1       0.43      0.80      0.56      1444

    accuracy                           0.92     23624
   macro avg       0.71      0.86      0.76     23624
weighted avg       0.95      0.92      0.93     23624


--- Running ABLATED feature set experiment for Diabetes ---

--- Training XGBoost with Ablated Features features for Diabetes ---
Train Accuracy (Optimal Thresh): 0.8881   Test Accuracy (Optimal Thresh): 0.8

## What the Final Ablation Shows

The final ablation experiment, where potentially leaky count-based features (`OBSERVATION_COUNT`, `DISTINCT_CONDITION_COUNT`, `DISTINCT_MEDICATION_COUNT`, `TOTAL_ENCOUNTERS`) were removed, provides critical insights into the true predictive power of our models:

1.  **Performance Drop:** For most targets, there is an expected drop in metrics like ROC-AUC and PR-AUC when these features are removed. This drop quantifies the extent to which the original models were relying on the leaked information.
2.  **Shift in Feature Importance:** After ablation, the feature importance tables reveal a shift towards more clinically relevant physiological measurements (e.g., `AVG_GLUCOSE`, `AVG_SYSTOLIC_BP`, `AVG_CHOLESTEROL`) and established risk factors (e.g., `AGE_AT_INDEX`, `AVG_WEIGHT`). This indicates that the models are now learning from genuinely predictive features that would be available prior to diagnosis.
3.  **More Realistic Performance:** The performance metrics from the ablated models, while lower than the full-feature models, are a more realistic representation of what can be achieved in a prospective, leakage-free prediction scenario. These metrics are more trustworthy for assessing the model's utility in early detection or risk stratification.
4.  **Guidance for Feature Engineering:** The results of the ablation guide future feature engineering efforts. It underscores the importance of temporal validity and avoids features that are consequences of the target outcome.

In [ ]:
# Feature Importance Output (Text Tables)

print("\n\n=== TOP 10 FEATURE IMPORTANCES ACROSS ALL EXPERIMENTS ===")

for key, fi_df in all_feature_importances.items():
    target_name, feature_set = key.split('_')
    print(f"\n--- {target_name} ({feature_set} Features) ---")
    print(fi_df.head(10).to_string(index=False))



=== TOP 10 FEATURE IMPORTANCES ACROSS ALL EXPERIMENTS ===

--- Diabetes (Full Features) ---
                        Feature  Importance
          num__TOTAL_ENCOUNTERS    0.423996
           cat__MARITAL_UNKNOWN    0.251244
              num__AGE_AT_INDEX    0.064112
         num__OBSERVATION_COUNT    0.039404
                cat__RACE_white    0.019534
  num__DISTINCT_CONDITION_COUNT    0.013226
       num__HAS_GLUCOSE_READING    0.012436
 num__DISTINCT_MEDICATION_COUNT    0.010260
   num__HAS_CHOLESTEROL_READING    0.007796
cat__ETHNICITY_central_american    0.006729

--- Diabetes (Ablated Features) ---
                     Feature  Importance
        cat__MARITAL_UNKNOWN    0.560142
           num__AGE_AT_INDEX    0.098289
             num__AVG_HEIGHT    0.069543
num__HAS_CHOLESTEROL_READING    0.055904
             num__AVG_WEIGHT    0.048067
             cat__RACE_white    0.020724
    num__HAS_GLUCOSE_READING    0.011271
                num__AVG_BMI    0.007008
      cat__ETHNI

## Model Saving for Deployment

This section demonstrates how to save the final (full-feature) XGBoost pipelines along with their metadata for potential deployment. Each model is saved using `joblib` and accompanied by a JSON file containing relevant details, as per the `MODEL_HANDOFF_GUIDE.md`.

In [ ]:
# Model Saving Cell

import os

# Create a directory for saved models if it doesn't exist
SAVE_DIR = 'saved_models'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Saving final (full-feature) XGBoost models to '{SAVE_DIR}'...")

for target_name, pipeline in full_feature_pipelines.items():
    model_filename = os.path.join(SAVE_DIR, f'xgb_full_features_{target_name.lower().replace(" ", "_")}.joblib')
    metadata_filename = os.path.join(SAVE_DIR, f'xgb_full_features_{target_name.lower().replace(" ", "_")}_metadata.json')

    # Save the pipeline
    joblib.dump(pipeline, model_filename)
    print(f"  Saved model for {target_name} to {model_filename}")

    # Generate and save metadata (example structure based on common requirements)
    metadata = {
        'model_name': f'XGBoost_FullFeatures_{target_name}',
        'target_variable': target_name,
        'model_type': 'XGBoostClassifier',
        'feature_set_used': 'Full Features',
        'numeric_features': NUMERIC_COLS_FULL,
        'categorical_features': CATEGORICAL_COLS,
        'pipeline_steps': [step[0] for step in pipeline.steps],
        'training_metrics': final_comparison_df[
            (final_comparison_df['Target'] == target_name) &
            (final_comparison_df['Feature_Set'] == 'Full Features')
        ].iloc[0].to_dict(),
        'description': f'XGBoost model predicting {target_name} using the full feature set (including count features).'
    }

    with open(metadata_filename, 'w') as f:
        json.dump(metadata, f, indent=4)
    print(f"  Saved metadata for {target_name} to {metadata_filename}")

print("\nAll specified models and metadata saved successfully.")

Saving final (full-feature) XGBoost models to 'saved_models'...
  Saved model for Diabetes to saved_models/xgb_full_features_diabetes.joblib
  Saved metadata for Diabetes to saved_models/xgb_full_features_diabetes_metadata.json
  Saved model for Hypertension to saved_models/xgb_full_features_hypertension.joblib
  Saved metadata for Hypertension to saved_models/xgb_full_features_hypertension_metadata.json
  Saved model for Coronary Heart Disease to saved_models/xgb_full_features_coronary_heart_disease.joblib
  Saved metadata for Coronary Heart Disease to saved_models/xgb_full_features_coronary_heart_disease_metadata.json
  Saved model for Stroke to saved_models/xgb_full_features_stroke.joblib
  Saved metadata for Stroke to saved_models/xgb_full_features_stroke_metadata.json
  Saved model for Asthma to saved_models/xgb_full_features_asthma.joblib
  Saved metadata for Asthma to saved_models/xgb_full_features_asthma_metadata.json

All specified models and metadata saved successfully.
